# MorphoNet-BT — 04: Architecture and Forward Validation
## Objective and scope
Implement the original proposed design and check shapes, branches and gradients with synthetic data and a small real MRI batch. This historical phase performs no repeated training, optimizer update, accuracy-based selection, THOP/MAC/FLOP study or Grad-CAM. It uses one total-loss backward pass and a separate morphology-loss backward pass on the retained graph, without an optimizer step. Splits and Preprocessing v1.0 remain frozen; only model settings are added.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.

**Model identity:** this phase concerns the original full model, later A5. The principal manuscript A4 uses `build_ablation("A4")` and fixed mean fusion, not AMGF.


## Original architecture overview
MRI → MSS → Stage 1 → Stage 2 → Stage 3 → Global/Local → AMGF → Stage 4 → GAP → Dropout → Classifier.
Stage 2 feeds MorphologyHead. Predicted boundary logits are resized to 16×16 and added to a score learned from local features within BAA. This connects morphology directly to classification, in addition to their shared MSS and stages 1/2. The connection is not detached; ground-truth masks and tumorBorder never enter forward. The design combines standard modules; this historical description does not establish component novelty or priority. This is the original full model (later A5), not principal A4.

## Imports, device and seed

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from pathlib import Path
from copy import deepcopy
import sys, json, hashlib, platform, inspect
PROJECT_ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
assert Path(sys.prefix).name.lower() == "dl", sys.executable
import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader
from IPython.display import display, Markdown
from src.data.dataset import BrainTumorDataset, verify_frozen_splits
from src.models.morphonet_bt import MorphoNetBT, MODEL_CONFIG
from src.models.validation import (validate_outputs, smoke_losses, SMOKE_LOSS_WEIGHTS,
    gradient_table, gradient_groups, tensor_statistics, shape_table, residual_checks)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT = PROJECT_ROOT / "results/architecture"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable, platform.python_version())
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, "Device:", device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))

## Preserve Preprocessing v1.0 and add model settings only

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
frozen_before = verify_frozen_splits(PROJECT_ROOT)
data_sources = [PROJECT_ROOT / "src/data" / name for name in ("preprocessing.py", "morphology.py", "dataset.py")]
source_hashes = {str(p.relative_to(PROJECT_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in data_sources}
approved_path = OUTPUT / "preprocessing_v1_approved.yaml"
approved = yaml.safe_load(approved_path.read_text(encoding="utf-8"))
config_path = PROJECT_ROOT / "configs/default.yaml"
config_text = config_path.read_text(encoding="utf-8")
config = yaml.safe_load(config_text)
assert {k: v for k, v in config.items() if k != "model"} == approved
if "model" in config:
    assert config["model"] == MODEL_CONFIG, "Existing model configuration differs; review required"
else:
    config_path.write_text(config_text + "\n" + yaml.safe_dump({"model": MODEL_CONFIG}, sort_keys=False), encoding="utf-8")
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
assert {k: v for k, v in config.items() if k != "model"} == approved
display(config["model"])
print("PASS: frozen split hashes and preprocessing settings preserved")

## Multi-Scale Stem (MSS)
Three stride-2 branches each produce 16 channels: 3×3 convolution; depthwise 5×5 then pointwise; depthwise 3×3 with dilation 2 then pointwise. Concatenation gives 48 channels, followed by a 1×1 projection to 32. With single-channel input, the depthwise operations in branches B/C each use one spatial filter before channel expansion. ConvNormAct uses BatchNorm and SiLU. Output: `[B,32,128,128]`.
## MFEBlock and channel gate
Expansion 1×1 (ratio 2) → depthwise 3×3 → linear pointwise with BatchNorm → SE-like channel gate → shortcut addition → SiLU. The gate uses GAP and two 1×1 layers with SiLU/Sigmoid, reduction 16 and at least four hidden channels. The shortcut is identity for matching channels and stride 1; otherwise it uses 1×1 convolution and BatchNorm at the required stride. The first block of each stage downsamples; remaining blocks use stride 1.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
residual_report = residual_checks(device)
display(residual_report)
residual_report.to_csv(OUTPUT / "residual_checks.csv", index=False)

## MorphologyHead and classification connection
Stage 2 `[B,80,32,32]` → 32-channel projection → depthwise 3×3 → three 1×1 predictors. Region/boundary raw logits are resized bilinearly to 256×256 with align_corners=False, without sigmoid, for BCEWithLogits. Distance logits are resized then passed through sigmoid to constrain values to [0,1]; this does not guarantee distance learning. Low-resolution boundary logits remain active during inference. Region/distance predictors and full-resolution maps are skipped when return_aux=False.
## Boundary-Aware Attention (BAA)
`A = sigmoid(G(F_local) + resize(boundary_logits_stage2))`; `F_out = F_local × (1+A)`. G uses depthwise 3×3 → SiLU → single-channel 1×1. A has shape `[B,1,16,16]` and broadcasts over channels. Boundary supervision trains the same predictor that guides classification. G receives classification supervision; A is not a validated tumor segmentation.
## Global/local paths and original AMGF
Global: depthwise 5×5 with dilation 2 (effective 9×9 field), then pointwise; not all-to-all attention. Local: depthwise 3×3 → BAA → pointwise. Both outputs are `[B,128,16,16]`. AMGF applies GAP to their sum, then 128→16→128 with SiLU/Sigmoid. Alpha is `[B,128,1,1]`; `F_fused = alpha*F_local + (1-alpha)*F_global`, without concatenation. A4 later replaces this original A5 fusion with the fixed arithmetic mean.
## Stage 4, classification and initialization
Two MFE blocks yield `[B,192,8,8]`, followed by GAP, flatten, dropout 0.30 and a linear layer producing three raw logits without softmax. Convolutions use Kaiming normal, fan_in and ReLU gain as an approximation for SiLU, with zero biases. BatchNorm scale/bias are 1/0; linear initialization is Xavier uniform with zero bias. Class order: 0=Meningioma, 1=Glioma, 2=Pituitary. Training starts from scratch without a pretrained backbone.

## Instantiate the model and obtain an initial parameter count
Flags provide a configurable architecture; this notebook does not run ablation experiments.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
model = MorphoNetBT(**config["model"]).to(device)
parameter_count = sum(p.numel() for p in model.parameters())
print("Basic parameter count:", parameter_count)
assert parameter_count < 1_000_000, "Parameter budget exceeded: stop and review"
initial_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
print("Forward signature:", inspect.signature(model.forward))
assert list(inspect.signature(model.forward).parameters) == ["image", "return_aux"]

## Synthetic forward and auxiliary shapes — MRI input only

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
x = torch.randn(2, 1, 256, 256, device=device, dtype=torch.float32)
model.eval()
with torch.no_grad():
    logits = model(x)
    synthetic = model(x, return_aux=True)
assert logits.shape == (2, 3) and torch.isfinite(logits).all()
assert torch.allclose(logits, synthetic["logits"], atol=1e-6, rtol=1e-5)
synthetic_shapes = validate_outputs(synthetic, 2)
display(pd.Series(synthetic_shapes))
print("Fusion gate range:", float(synthetic["fusion_gate"].min()), float(synthetic["fusion_gate"].max()))
print("Distance range:", float(synthetic["distance"].min()), float(synthetic["distance"].max()))
model.train()
with torch.no_grad(): train_synthetic = model(x, return_aux=True)
validate_outputs(train_synthetic, 2)
print("PASS: eval MRI-only, eval auxiliary, and train-mode forward")
del synthetic, train_synthetic, logits

## Tensor shapes from actual forward hooks

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model.eval()
shapes = shape_table(model, x)
display(shapes)
shapes.to_csv(OUTPUT / "module_shapes.csv", index=False)
expected_stage_shapes = {"MSS": [2,32,128,128], "Stage 1": [2,48,64,64], "Stage 2": [2,80,32,32],
                         "Stage 3": [2,128,16,16], "Stage 4": [2,192,8,8], "Classifier": [2,3]}
for module, expected in expected_stage_shapes.items():
    actual = json.loads(shapes.loc[shapes.module.eq(module), "output_shape"].iloc[0])
    assert actual == expected, f"Unexpected shape at {module}"

## Small real batch from fold 1 training data
Use the approved v1.0 dataset settings unchanged. No Test results or accuracy are computed.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
dataset = BrainTumorDataset(PROJECT_ROOT / "splits/fold_1_train.csv", mode="train", config=config, seed=SEED)
dataset.set_epoch(0)
loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(SEED))
cpu_batch = next(iter(loader))
batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in cpu_batch.items()}
assert batch["image"].shape == (2, 1, 256, 256) and batch["image"].dtype == torch.float32
model.train()
outputs = model(batch["image"], return_aux=True)
real_shapes = validate_outputs(outputs, 2)
display(pd.Series(real_shapes))
print("Sample IDs:", batch["sample_id"])
print("Labels:", batch["label"].tolist())

## Loss compatibility — untrained smoke-test values only
Classification CE; BCEWithLogits plus soft Dice for region and boundary; SmoothL1 for distance. Dice is calculated per sample then averaged over the batch with epsilon=1e-6; logits are not sigmoid-transformed twice. `L_morph = L_region + L_boundary + 0.5*L_distance`; `L_total = L_cls + 0.3*L_morph`. These untrained-model checks are not experimental performance comparisons.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
losses = smoke_losses(outputs, batch, SMOKE_LOSS_WEIGHTS)
loss_values = {k: float(v.detach().cpu()) for k, v in losses.items()}
display(pd.Series(loss_values))
pd.Series(loss_values, name="untrained_smoke_value").to_csv(OUTPUT / "smoke_losses.csv")

## One total-loss backward pass — no optimizer step
Retain the graph only for the following separate diagnostic. Do not repeat forward or update weights.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model.zero_grad(set_to_none=True)
losses["total"].backward(retain_graph=True)
total_gradients = gradient_table(model)
total_groups = gradient_groups(total_gradients)
assert total_gradients.present.all(), "An intended parameter received no total-loss gradient"
assert total_gradients.finite.all(), "Nonfinite total gradient"
assert total_groups.gradient_norm.gt(0).all(), "A required module has zero gradient"
display(total_groups)
total_gradients.to_csv(OUTPUT / "gradient_flow_parameters.csv", index=False)
total_groups.to_csv(OUTPUT / "gradient_flow.csv", index=False)

## Morphology-only gradients in the shared encoder
Clear gradients and backpropagate morphology loss on the retained graph without updating weights. Direct gradients are expected in MSS, stages 1/2 and MorphologyHead, not stages 3/4, local BAA, AMGF or classifier; the latter learn through classification in the total loss.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model.zero_grad(set_to_none=True)
losses["morphology"].backward()
aux_gradients = gradient_table(model)
aux_groups = gradient_groups(aux_gradients)
shared = aux_groups[aux_groups.module.isin(["stem", "stage1", "stage2", "morphology_head"])]
assert shared.with_gradient.eq(shared.parameter_tensors).all()
assert shared.all_finite.all() and shared.gradient_norm.gt(0).all()
assert not aux_gradients.loc[aux_gradients.parameter.str.startswith("classifier."), "present"].any()
display(aux_groups)
aux_gradients.to_csv(OUTPUT / "auxiliary_gradient_parameters.csv", index=False)
aux_groups.to_csv(OUTPUT / "auxiliary_gradient_flow.csv", index=False)
print("PASS: morphology supervision reaches the shared encoder")

## Untrained attention and gate checks
These numbers check numerical validity, not tumor localization or biological interpretation.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
attention_statistics = {"boundary_attention": tensor_statistics(outputs["boundary_attention"]),
                        "fusion_gate": tensor_statistics(outputs["fusion_gate"])}
display(pd.DataFrame(attention_statistics).T)
pd.DataFrame(attention_statistics).T.to_csv(OUTPUT / "untrained_attention_gate_statistics.csv")
weights_unchanged = all(torch.equal(p.detach().cpu(), initial_state[name]) for name, p in model.named_parameters())
assert weights_unchanged, "Weights changed during diagnostics"
# Training-mode BatchNorm updates buffers even without an optimizer; restore all initial state.
model.zero_grad(set_to_none=True)
model.load_state_dict(initial_state)
assert all(torch.equal(value.detach().cpu(), initial_state[name]) for name, value in model.state_dict().items())
del losses, outputs
print("PASS: parameters unchanged; BatchNorm buffers restored to initial state")

## MRI-only inference and rejection of ground-truth arguments
In eval mode, return_aux=False and True yield identical logits. The interface does not accept mask, tumorBorder or targets.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
model.eval()
with torch.inference_mode():
    image_only_logits = model(x)
    full_outputs = model(x, return_aux=True)
assert image_only_logits.shape == (2, 3)
assert torch.isfinite(image_only_logits).all()
assert torch.allclose(image_only_logits, full_outputs["logits"], atol=1e-6, rtol=1e-5)
try:
    model(x, tumorMask=torch.zeros_like(x))
except TypeError:
    print("PASS: public forward rejects a ground-truth mask argument")
else:
    raise AssertionError("Model unexpectedly accepts ground-truth mask")
assert verify_frozen_splits(PROJECT_ROOT) == frozen_before
assert yaml.safe_load(config_path.read_text(encoding="utf-8")) == config
assert {k: v for k, v in config.items() if k != "model"} == approved
assert source_hashes == {str(p.relative_to(PROJECT_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in data_sources}
print("PASS: MRI-only inference, frozen splits, and Preprocessing v1.0")

## Save the validation report without training checkpoints

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
report = {
    "status": "PASS", "seed": SEED, "device": str(device), "torch": torch.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    "parameter_count_basic": parameter_count, "synthetic_shapes": synthetic_shapes, "real_shapes": real_shapes,
    "real_sample_ids": batch["sample_id"], "real_labels": batch["label"].tolist(),
    "smoke_losses_untrained": loss_values, "loss_weights": SMOKE_LOSS_WEIGHTS,
    "attention_gate_untrained": attention_statistics,
    "total_gradient_all_parameters_present_finite": True, "auxiliary_reaches_shared_encoder": True,
    "total_backward_calls": 1, "auxiliary_backward_calls": 1, "optimizer_steps": 0,
    "weights_unchanged": weights_unchanged, "batchnorm_buffers_restored": True,
    "mri_only_inference": True, "ground_truth_mask_argument_rejected": True,
    "preprocessing_unchanged": True, "frozen_splits_unchanged": True,
    "formal_complexity_study": "NOT RUN", "training": "NOT RUN", "phase_5": "NOT STARTED"}
(OUTPUT / "validation_results.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
(OUTPUT / "frozen_input_hashes.json").write_text(json.dumps({"splits": frozen_before, "preprocessing_sources": source_hashes}, indent=2), encoding="utf-8")
summary_text = f"""MorphoNet-BT — Phase 4: architecture validation only.
Result: PASS. Device: {device}. Initial parameter count: {parameter_count}.
Path: 1×256×256 → MSS 32×128×128 → Stage1 48×64×64 → Stage2 80×32×32 → Stage3 128×16×16 → Global/Local/AMGF → Stage4 192×8×8 → GAP → Dropout .30 → 3 logits.
MFE blocks: 2/2/3/2, expansion=2, SE-like reduction=16.
MorphologyHead: Stage2 32-channel bottleneck, region/boundary logits and sigmoid distance, B×1×256×256.
BAA: sigmoid(local_score + resized learned boundary logits), then F×(1+A), A is B×1×16×16; no GT input.
Global: depthwise 5×5 dilation2 then pointwise. Local: depthwise3×3, BAA, pointwise.
AMGF: pooled sum, MLP 128→16→128, alpha B×128×1×1.
Synthetic eval/train forward, real Train batch size2 and MRI-only inference passed.
Untrained diagnostic losses: {json.dumps(loss_values)}.
Total-loss backward reached all required parameters/modules. L_morph reached MSS, Stage1/2 and MorphologyHead; direct gradients in later stages are not expected.
Backward calls: total=1, auxiliary=1. Optimizer steps=0. Weights unchanged; BN buffers restored.
No NaN/Inf or shape errors. Distance, attention and alpha in [0,1].
Preprocessing v1.0 and splits unchanged; only model settings added to default.yaml.
Reusing the supervised boundary predictor for BAA directly connects morphology to classification without inference-time ground truth.
Untrained attention/gates are not biological explanations. Standard-module novelty or priority is not established.
MACs/FLOPs/THOP, training, ablation and Grad-CAM: NOT RUN.
Complexity analysis awaits researcher approval; Phase 5 has not started.
"""
(OUTPUT / "architecture_validation.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)

## Historical conclusion
Shapes, branches, maps, MRI-only inference and gradients were checked without training. Morphology guidance is explicit and derived from MRI alone. Boundary predictions originate at 32×32 and are resized; fine boundary recovery remains an empirical question. The boundary predictor must be counted in inference complexity, although ground-truth masks are unnecessary. Low parameter count does not establish accuracy, and the FLOP target had not yet been tested. The historical workflow stopped for review before Phase 5.
**Recorded historical results:** CUDA; 579,599 parameters in the original full model; synthetic and real batches of 2 passed. Logits `[2,3]`, maps `[2,1,256,256]`, attention `[2,1,16,16]`, alpha `[2,128,1,1]`. All total-loss gradients were present and finite; morphology loss reached the shared encoder. No optimizer steps or changes to weights/frozen settings. These are retained historical statements, not Phase 18 executions.